<a href="https://colab.research.google.com/github/Rohan-Bha/Special-Topcis-MIS-IntroToAI2026/blob/main/Ad_Optimization_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import pandas as pd
import json
import os
import math
from collections import defaultdict

In [13]:
TOTAL_DAILY_BUDGET = 1000.00      # Total budget to allocate each day
MIN_CHANNEL_FLOOR  = 0.10         # No channel gets less than 10% of budget
MAX_SHIFT_PER_DAY  = 0.20         # Max ±20% reallocation per step
EXPLORE_RATE       = 0.10         # 10% explore bonus to keep learning signal
LOOKBACK_DAYS      = 7            # Days of history used for scoring

In [14]:
SYSTEM_PROMPT = """
You are an Ad Budget Optimization Agent for a performance marketing team.

ROLE
You receive daily ad performance data (spend, impressions, clicks, conversions)
across three channels: Search, Social, and Display. Your job is to propose the
next day's budget split across those channels to maximize total conversions
while managing cost-per-acquisition (CPA).

INPUTS
- Historical performance table (last N days per channel)
- Current equal-split allocation as a baseline

OUTPUTS
- Proposed budget allocation (dollars) for each channel
- Short rationale string (≤ 30 words) for each channel's change
- Overall decision summary

RULES / GUARDRAILS
1. Never allocate 0% to any channel for more than 2 consecutive days.
2. Cap any single-day budget change at ±20% of the previous allocation.
3. Maintain a minimum floor of 10% of total budget per channel.
4. Log every decision with a reason string (e.g., "Search +12% due to CVR 4.2%").
5. Privacy: do not surface user-level or PII data; work only with aggregates.
6. Brand tone: be factual, concise, and numbers-driven in all rationale strings.

EVALUATION METRIC
Primary   → Total conversions over the evaluation window
Secondary → Average CPA (lower is better)
Guard     → No channel starved (floor respected, zero-day rule upheld)
"""


In [16]:
def read_performance_data(csv_path: str) -> dict:
    """
    Reads the CSV using pandas and returns a dict keyed by (date, channel).
    Also returns sorted list of unique dates and channels.
    """
    df = pd.read_csv(csv_path)

    # Ensure correct types
    df["spend"]       = df["spend"].astype(float)
    df["impressions"] = df["impressions"].astype(int)
    df["clicks"]      = df["clicks"].astype(int)
    df["conversions"] = df["conversions"].astype(int)
    df["date"]        = df["date"].astype(str)

    records  = df.to_dict(orient="records")
    dates    = sorted(df["date"].unique().tolist())
    channels = sorted(df["channel"].unique().tolist())
    by_key   = {(r["date"], r["channel"]): r for r in records}

    return {"records": records, "dates": dates, "channels": channels, "by_key": by_key}

In [17]:
def compute_channel_scores(data: dict, as_of_date: str, lookback: int = LOOKBACK_DAYS) -> dict:
    """
    For each channel computes rolling metrics over the last `lookback` days
    ending at (but not including) as_of_date.

    Returns dict: channel → {cvr, ctr, cpa, total_conversions, total_spend, score}
    score = CVR * (1 / CPA_normalized)  — higher is better
    """
    dates    = [d for d in data["dates"] if d < as_of_date]
    window   = dates[-lookback:] if len(dates) >= lookback else dates
    channels = data["channels"]
    by_key   = data["by_key"]

    stats = {}
    for ch in channels:
        spend = clicks = conversions = impressions = 0
        for d in window:
            r = by_key.get((d, ch), {})
            spend       += r.get("spend",       0)
            clicks      += r.get("clicks",      0)
            conversions += r.get("conversions", 0)
            impressions += r.get("impressions", 0)

        cvr = conversions / clicks      if clicks      > 0 else 0
        ctr = clicks      / impressions if impressions > 0 else 0
        cpa = spend       / conversions if conversions > 0 else float("inf")

        stats[ch] = {
            "cvr":               cvr,
            "ctr":               ctr,
            "cpa":               cpa,
            "total_conversions": conversions,
            "total_spend":       spend,
            "window_days":       len(window),
        }

    # Normalize CPA so lower CPA → higher sub-score (avoid div-by-zero)
    valid_cpas = [v["cpa"] for v in stats.values() if math.isfinite(v["cpa"])]
    max_cpa    = max(valid_cpas) if valid_cpas else 1.0

    for ch in channels:
        cpa   = stats[ch]["cpa"]
        cvr   = stats[ch]["cvr"]
        cpa_n = (cpa / max_cpa) if math.isfinite(cpa) else 1.0
        stats[ch]["score"] = cvr * (1 - cpa_n * 0.5)   # blend CVR & CPA efficiency

    return stats


In [18]:
def propose_budget(
    scores:       dict,
    prev_alloc:   dict,
    total_budget: float = TOTAL_DAILY_BUDGET,
    min_floor:    float = MIN_CHANNEL_FLOOR,
    max_shift:    float = MAX_SHIFT_PER_DAY,
    explore_rate: float = EXPLORE_RATE,
    zero_day_counts: dict = None,
) -> tuple[dict, list[str]]:
    """
    Heuristic + explore/exploit budget allocation.

    1. Score-proportional allocation (exploit)
    2. Add explore bonus to underperformers
    3. Clip to ±20% shift from previous day
    4. Enforce floor (10% minimum)
    5. Renormalize to total_budget
    6. Guardrail: if channel has been at 0% for ≥2 days, force floor
    """
    channels = list(scores.keys())
    zero_day_counts = zero_day_counts or {ch: 0 for ch in channels}
    rationale = []

    # ── Step 1: raw score-proportional weights ──
    total_score = sum(max(s["score"], 0.001) for s in scores.values())
    raw = {ch: (max(scores[ch]["score"], 0.001) / total_score) for ch in channels}

    # ── Step 2: explore bonus — smallest-score channel gets a nudge ──
    min_ch = min(channels, key=lambda c: scores[c]["score"])
    for ch in channels:
        if ch == min_ch:
            raw[ch] = raw[ch] + explore_rate * (1 - raw[ch])

    # Renorm after explore
    s = sum(raw.values())
    raw = {ch: raw[ch] / s for ch in channels}

    # ── Step 3: convert to dollars and clip shift ──
    proposed = {}
    for ch in channels:
        target    = raw[ch] * total_budget
        prev      = prev_alloc.get(ch, total_budget / len(channels))
        max_up    = prev * (1 + max_shift)
        max_down  = prev * (1 - max_shift)
        clipped   = max(max_down, min(target, max_up))
        proposed[ch] = clipped

    # ── Step 4: enforce floor ──
    floor_dollars = min_floor * total_budget
    for ch in channels:
        if proposed[ch] < floor_dollars:
            proposed[ch] = floor_dollars
        # Guardrail: never-zero rule
        if zero_day_counts.get(ch, 0) >= 2 and proposed[ch] < floor_dollars:
            proposed[ch] = floor_dollars

    # ── Step 5: renormalize to exact total_budget ──
    total_prop = sum(proposed.values())
    scale = total_budget / total_prop
    proposed = {ch: round(v * scale, 2) for ch, v in proposed.items()}

    # Fix rounding so sum is exact
    diff = total_budget - sum(proposed.values())
    top_ch = max(channels, key=lambda c: proposed[c])
    proposed[top_ch] = round(proposed[top_ch] + diff, 2)

    # ── Step 6: build rationale strings ──
    for ch in channels:
        prev   = prev_alloc.get(ch, total_budget / len(channels))
        delta  = proposed[ch] - prev
        pct    = (delta / prev) * 100 if prev > 0 else 0
        cvr_str = f"{scores[ch]['cvr']*100:.1f}%"
        cpa_str = f"${scores[ch]['cpa']:.1f}" if math.isfinite(scores[ch]['cpa']) else "N/A"
        direction = "up" if pct >= 0 else "down"
        reason = (
            f"{ch} {direction} {abs(pct):.1f}% | "
            f"CVR {cvr_str}, CPA {cpa_str}, "
            f"score {scores[ch]['score']:.4f}"
        )
        rationale.append(reason)

    return proposed, rationale


In [19]:
def run_agent(csv_path: str, verbose: bool = True) -> list[dict]:
    """
    Simulates one decision per day across the dataset.
    Returns list of daily decision dicts for evaluation.
    """
    data     = read_performance_data(csv_path)
    dates    = data["dates"]
    channels = data["channels"]
    by_key   = data["by_key"]

    # Starting allocation: equal split
    prev_alloc = {ch: TOTAL_DAILY_BUDGET / len(channels) for ch in channels}
    zero_day_counts = {ch: 0 for ch in channels}

    decisions = []

    print("=" * 65)
    print("  AD OPTIMIZATION AGENT  —  Decision Log")
    print(f"  System: {SYSTEM_PROMPT.strip().splitlines()[1].strip()}")
    print("=" * 65)

    for i, date in enumerate(dates):
        if i == 0:
            # No history yet — use equal split, skip scoring
            alloc   = {ch: round(TOTAL_DAILY_BUDGET / len(channels), 2) for ch in channels}
            reasons = [f"{ch}: equal split (day 1, no history)" for ch in channels]
            scores  = {ch: {"cvr": 0, "ctr": 0, "cpa": 0, "score": 0, "total_conversions": 0, "total_spend": 0} for ch in channels}
        else:
            scores  = compute_channel_scores(data, as_of_date=date)
            alloc, reasons = propose_budget(
                scores, prev_alloc,
                zero_day_counts=zero_day_counts
            )

        # Actual actuals for this date
        actuals = {}
        for ch in channels:
            r = by_key.get((date, ch), {})
            actuals[ch] = {
                "spend":       r.get("spend",       0),
                "conversions": r.get("conversions", 0),
                "clicks":      r.get("clicks",      0),
                "impressions": r.get("impressions", 0),
            }

        total_conversions = sum(actuals[ch]["conversions"] for ch in channels)
        total_spend       = sum(actuals[ch]["spend"]       for ch in channels)
        avg_cpa           = total_spend / total_conversions if total_conversions > 0 else float("inf")

        decisions.append({
            "date":              date,
            "proposed_alloc":    alloc,
            "actual_actuals":    actuals,
            "scores":            scores,
            "rationale":         reasons,
            "total_conversions": total_conversions,
            "total_spend":       total_spend,
            "avg_cpa":           avg_cpa,
        })

        if verbose:
            print(f"\n📅  {date}")
            print(f"  Proposed Allocation:")
            for ch in channels:
                pct = alloc[ch] / TOTAL_DAILY_BUDGET * 100
                print(f"    {ch:<8} ${alloc[ch]:>7.2f}  ({pct:.1f}%)")
            print(f"  Rationale:")
            for r in reasons:
                print(f"    → {r}")
            print(f"  Actual outcome → {total_conversions} conversions | Avg CPA ${avg_cpa:.2f}")

        # Update prev_alloc and zero-day tracker
        prev_alloc = alloc
        for ch in channels:
            pct = alloc[ch] / TOTAL_DAILY_BUDGET
            zero_day_counts[ch] = 0 if pct > MIN_CHANNEL_FLOOR else zero_day_counts[ch] + 1

    return decisions


In [20]:
def evaluate(decisions: list[dict]) -> dict:
    """
    Computes summary evaluation metrics across all days.
    """
    total_conv  = sum(d["total_conversions"] for d in decisions)
    total_spend = sum(d["total_spend"]       for d in decisions)
    avg_cpa     = total_spend / total_conv if total_conv > 0 else float("inf")

    # Conversion trend: compare first half vs second half
    mid   = len(decisions) // 2
    conv_first  = sum(d["total_conversions"] for d in decisions[:mid])
    conv_second = sum(d["total_conversions"] for d in decisions[mid:])
    trend       = (conv_second - conv_first) / conv_first * 100 if conv_first > 0 else 0

    # Floor violations
    floor_violations = 0
    for d in decisions:
        alloc = d["proposed_alloc"]
        for ch, amt in alloc.items():
            if amt / TOTAL_DAILY_BUDGET < MIN_CHANNEL_FLOOR - 0.001:
                floor_violations += 1

    channel_totals = defaultdict(lambda: {"conversions": 0, "spend": 0})
    for d in decisions:
        for ch, act in d["actual_actuals"].items():
            channel_totals[ch]["conversions"] += act["conversions"]
            channel_totals[ch]["spend"]       += act["spend"]

    metrics = {
        "total_conversions":        total_conv,
        "total_spend":              round(total_spend, 2),
        "avg_cpa":                  round(avg_cpa, 2),
        "conversion_trend_pct":     round(trend, 1),
        "floor_violations":         floor_violations,
        "days_evaluated":           len(decisions),
        "channel_breakdown":        {ch: dict(v) for ch, v in channel_totals.items()},
    }

    print("\n" + "=" * 65)
    print("  EVALUATION SUMMARY")
    print("=" * 65)
    print(f"  Days evaluated    : {metrics['days_evaluated']}")
    print(f"  Total conversions : {metrics['total_conversions']}")
    print(f"  Total spend       : ${metrics['total_spend']:,.2f}")
    print(f"  Average CPA       : ${metrics['avg_cpa']:.2f}")
    print(f"  Conv trend (2H-1H): {metrics['conversion_trend_pct']:+.1f}%")
    print(f"  Floor violations  : {metrics['floor_violations']}")
    print("\n  Channel Breakdown:")
    for ch, v in metrics["channel_breakdown"].items():
        cpa = v["spend"] / v["conversions"] if v["conversions"] > 0 else 0
        print(f"    {ch:<8}  {v['conversions']} conv  |  ${v['spend']:,.0f} spend  |  CPA ${cpa:.2f}")
    print("=" * 65)

    return metrics

In [21]:
CSV_PATH = "ad_optimization_mock_data.csv"  # ← change this to your CSV filename if different

decisions = run_agent(CSV_PATH, verbose=True)
metrics   = evaluate(decisions)

# Save decision log
output_path = "decision_log.json"
with open(output_path, "w") as f:
    safe = json.loads(json.dumps(decisions, default=str))
    json.dump(safe, f, indent=2)
print(f"\n  Decision log saved → {output_path}")



  AD OPTIMIZATION AGENT  —  Decision Log
  System: 

📅  2023-01-01
  Proposed Allocation:
    Display  $ 333.33  (33.3%)
    Search   $ 333.33  (33.3%)
    Social   $ 333.33  (33.3%)
  Rationale:
    → Display: equal split (day 1, no history)
    → Search: equal split (day 1, no history)
    → Social: equal split (day 1, no history)
  Actual outcome → 15 conversions | Avg CPA $20.00

📅  2023-01-02
  Proposed Allocation:
    Display  $ 375.00  (37.5%)
    Search   $ 312.50  (31.2%)
    Social   $ 312.50  (31.2%)
  Rationale:
    → Display up 12.5% | CVR 10.0%, CPA $20.0, score 0.0500
    → Search down 6.2% | CVR 10.0%, CPA $20.0, score 0.0500
    → Social down 6.2% | CVR 10.0%, CPA $20.0, score 0.0500
  Actual outcome → 17 conversions | Avg CPA $18.53

📅  2023-01-03
  Proposed Allocation:
    Display  $ 345.65  (34.6%)
    Search   $ 332.91  (33.3%)
    Social   $ 321.44  (32.1%)
  Rationale:
    → Display down 7.8% | CVR 9.8%, CPA $20.4, score 0.0488
    → Search up 6.5% | CVR 10.7%, C